# Mallory, Hardcastle et al. (2021) on dandiset 000053: data check and behaviour

**Paper.** Mallory CS, Hardcastle K, Campbell MG, Attinger A, Low IIC, Raymond JL, Giocomo LM
(2021). Mouse entorhinal cortex encodes a diverse repertoire of self-motion signals. *Nature
Communications* 12, 671. [doi:10.1038/s41467-021-20936-8](https://doi.org/10.1038/s41467-021-20936-8).
The paper is open access; this attempt used its full text (Europe PMC, PMC7844029) and its
Supplementary Information.

**Dandiset.** [DANDI:000053](https://dandiarchive.org/dandiset/000053/0.210819.0345), version
0.210819.0345. The paper cites no DANDI version (its data statement points to figshare). This is
the only published version, so it is pinned; the cell below compares it with the draft.

**What the dandiset holds.** Freely moving tetrode sessions of 25 mice (body position, speed,
azimuthal head direction and velocity; no pitch or roll), and head-fixed Neuropixels sessions of
9 mice in virtual reality (track position, speed, eye position and velocity), some of them twice.

**Notebooks.** This one checks the data and tests the claims that need no model:
`02_head_fixed_ln.ipynb` fits the LN model to the head-fixed cells and `03_freely_moving_ln.ipynb`
to the freely moving cells. Each claim's section is in the notebook named below.

| Claim | Notebook | Status |
|---|---|---|
| vr-dataset-size | 01 | tested |
| of-cohorts-by-count | 01 | tested |
| fig1b-ahv-percentiles | 01 | tested |
| fig4fghi-eye-percentiles | 01 | tested |
| figs12-eye-behavior-mi | 01 | tested |
| fig4j-eye-fractions, fig4j-body-fractions, figs3c-false-positives-vr, fig4k-conjunctive-eye | 02 | tested |
| fig5-eye-position-preference, fig5-eye-position-stability | 02 | tested |
| fig6-eye-velocity-shapes, fig6-eye-velocity-direction, fig6-velocity-stability | 02 | tested |
| fig7ab-contributions-vr, fig7c-llh-vr, fig7e-no-clustering | 02 | tested |
| fig1c-cohort1-fractions, fig1c-ahv-cohort2, figs3b-false-positives-of | 03 | tested |
| fig2c-ahv-shapes, fig2cv-ahv-stability, fig3ab-contributions-of, fig3c-llh-of | 03 | tested |
| fig1b-pitch-roll-percentiles, fig1c-pitch-roll-fractions, fig1d-conjunctive-cohort1 | none | blocked: data missing (no IMU data) |
| fig2ab-pitch-roll-tuning, fig3-pitch-roll-contributions, fig3e-head-no-clustering | none | blocked: data missing (no IMU data) |
| fig8-landmarks | none | blocked: data missing |
| fig2c-ahv-direction, vr-hemispheres | none | blocked: metadata insufficient (no hemisphere) |

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
from scipy import stats

import mec

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

api = f"https://api.dandiarchive.org/api/dandisets/{mec.DANDISET_ID}"
versions = requests.get(f"{api}/versions/").json()["results"]
print("Versions:", [(v["version"], v["asset_count"], v["size"], v["modified"][:10]) for v in versions])

def asset_list(version):
    out, url = [], f"{api}/versions/{version}/assets/?page_size=1000"
    while url:
        page = requests.get(url).json()
        out += page["results"]
        url = page["next"]
    return {a["path"]: (a["size"], a.get("blob") or a["asset_id"]) for a in out}

pub, draft = asset_list(mec.DANDISET_VERSION), asset_list("draft")
print("Published assets:", len(pub), "; draft assets:", len(draft))
print("Paths only in one of them:", sorted(set(pub) ^ set(draft)))
print("Same paths with a different size:", [p for p in set(pub) & set(draft) if pub[p][0] != draft[p][0]])
meta_pub = requests.get(f"{api}/versions/{mec.DANDISET_VERSION}/").json()
meta_draft = requests.get(f"{api}/versions/draft/").json()
print("Metadata fields that differ:", sorted(k for k in set(meta_pub) | set(meta_draft)
                                             if meta_pub.get(k) != meta_draft.get(k)))
print("Assets summary at the pinned version:", {k: meta_pub["assetsSummary"][k] for k in ("numberOfFiles", "numberOfSubjects")},
      "species", [s["name"] for s in meta_pub["assetsSummary"]["species"]])
print("Description:", meta_pub["description"])

## Data check

The files at the pinned version, then the freely moving sessions.

In [ ]:
assets = mec.asset_table()
url = {a["path"]: a["url"] for a in assets}
kind = lambda p: "head-fixed behavior+ecephys" if "+ecephys" in p else ("head-fixed behavior" if "/sub-np" in p else "freely moving behavior")
inv = pd.DataFrame([{"path": a["path"], "kind": kind(a["path"]), "GB": a["size"] / 1e9} for a in assets])
print(inv.groupby("kind").agg(files=("path", "size"), GB=("GB", "sum")).round(1).to_string())

t0 = time.time()
of_paths = inv[inv.kind == "freely moving behavior"].path.tolist()
of = mec.load_many(mec.load_open_field, [url[p] for p in of_paths], workers=16)
for p, s in zip(of_paths, of):
    s["path"] = p
    s["group"] = mec.group_of(s["subject"])
print(f"loaded {len(of)} freely moving sessions in {time.time() - t0:.0f} s")

In [ ]:
ofi = pd.DataFrame([{
    "subject": s["subject"], "group": s["group"], "identifier": s["identifier"], "date": s["session_start"],
    "arena": mec.arena_side(s), "units": len(s["spikes"]), "samples": len(s["t"]),
    "minutes": (s["t"][-1] - s["t"][0]) / 60, "dt_ms": 1000 * np.median(np.diff(s["t"])),
    "x_extent": np.nanmax(s["x"]) - np.nanmin(s["x"]), "y_extent": np.nanmax(s["y"]) - np.nanmin(s["y"]),
    "nan_frac": float(np.mean(~np.isfinite(s["x"]))), "spikes_in_range": float(np.mean(
        [np.mean((sp >= s["t"][0]) & (sp <= s["t"][-1] + mec.DT)) for sp in s["spikes"]])),
} for s in of])
print("Freely moving: sessions", len(ofi), "; subjects", ofi.subject.nunique(), "; units", ofi.units.sum())
print("Species:", sorted({s["species"] for s in of}), "; descriptions:", sorted({s["description"] for s in of}))
print("Series:", sorted({tuple(s["behavior"]) for s in of}))
print("Stated units:", sorted({tuple(sorted(s["units_of"].items())) for s in of}))
print("\nBy subject:")
print(ofi.groupby(["group", "subject"]).agg(sessions=("units", "size"), units=("units", "sum"),
      arenas=("arena", lambda a: sorted(set(a))), first=("date", "min"), last=("date", "max"),
      minutes=("minutes", "median"), x_extent=("x_extent", "max"), y_extent=("y_extent", "max")).round(1).to_string())
print("\nSampling interval (ms), all sessions: median", ofi.dt_ms.median(), "range", ofi.dt_ms.min(), ofi.dt_ms.max())
print("Session length (min): range", round(ofi.minutes.min(), 1), "to", round(ofi.minutes.max(), 1))
print("Fraction of samples without position: max", ofi.nan_frac.max().round(4))
print("Fraction of spikes inside the behaviour samples: min", ofi.spikes_in_range.min().round(4))
print("\nArena size in the file against the extent of positions (cm):")
print(ofi.groupby("arena").agg(sessions=("units", "size"), x_extent_median=("x_extent", "median"),
      y_extent_median=("y_extent", "median"), x_extent_max=("x_extent", "max")).round(1).to_string())
spd = np.concatenate([s["Bs"] for s in of]); hd = np.concatenate([s["Ha"] for s in of]); ahv = np.concatenate([s["Hadot"] for s in of])
print("\nBody speed (cm/s) percentiles 0, 1, 50, 99, 100:", np.nanpercentile(spd, [0, 1, 50, 99, 100]).round(1))
print("Head direction (file says radians) min, max:", np.nanmin(hd).round(3), np.nanmax(hd).round(3))
print("Azimuthal head velocity, converted to deg/s, percentiles 0.1, 50, 99.9:", np.nanpercentile(ahv, [0.1, 50, 99.9]).round(1))
d_hd = np.concatenate([np.degrees(np.angle(np.exp(1j * np.diff(s["Ha"])))) / mec.DT for s in of])
ok = np.isfinite(d_hd) & np.isfinite(ahv[:len(d_hd)])
print("Correlation of the AHV series with the change in head direction per sample:",
      np.corrcoef(np.concatenate([s["Hadot"][1:] for s in of])[ok], d_hd[ok])[0, 1].round(3))
sp_calc = np.concatenate([np.hypot(np.diff(s["x"]), np.diff(s["y"])) / mec.DT for s in of])
sp_file = np.concatenate([s["Bs"][1:] for s in of])
ok = np.isfinite(sp_calc) & np.isfinite(sp_file) & (sp_calc < 200)
print("Body speed against position change: correlation", np.corrcoef(sp_calc[ok], sp_file[ok])[0, 1].round(3),
      "; median ratio", np.median(sp_file[ok] / np.maximum(sp_calc[ok], 1e-6)).round(3))

In [ ]:
# Recordings deposited more than once: identical spike trains or identical position traces.
def fingerprint(s):
    return (len(s["t"]), round(float(np.nansum(s["x"])), 3), sum(len(x) for x in s["spikes"]))
fp = pd.Series({s["path"]: fingerprint(s) for s in of})
dups = fp[fp.duplicated(keep=False)]
print("Freely moving files sharing a fingerprint (samples, sum of x, spikes):", len(dups))
print(dups.sort_values().to_string() if len(dups) else "none")
spk = {}
for s in of:
    for i, x in enumerate(s["spikes"]):
        if len(x) > 50:
            spk.setdefault((len(x), round(float(x[:50].sum()), 6)), []).append((s["path"], i))
same = {k: v for k, v in spk.items() if len(v) > 1}
print("Units whose spike trains repeat in another unit or file:", sum(len(v) for v in same.values()))
for k, v in list(same.items())[:10]:
    print("  ", v)
# Coverage: fraction of 5 cm bins of the occupied extent visited (Methods: sessions with >= 70%)
def coverage(s):
    ok = np.isfinite(s["x"]) & np.isfinite(s["y"])
    x, y = s["x"][ok] - np.min(s["x"][ok]), s["y"][ok] - np.min(s["y"][ok])
    h, _, _ = np.histogram2d(x, y, [np.arange(0, np.ptp(x) + 5, 5), np.arange(0, np.ptp(y) + 5, 5)])
    return float(np.mean(h > 0))
cov = np.array([coverage(s) for s in of])
print(f"\nCoverage of the occupied rectangle in 5 cm bins: median {np.median(cov):.2f}, min {cov.min():.2f}, "
      f"sessions below 0.70: {(cov < 0.7).sum()}")

### Head-fixed files

In [ ]:
vr_paths = inv[inv.kind != "freely moving behavior"].path.tolist()
with ThreadPoolExecutor(16) as pool:
    vrd = dict(zip(vr_paths, pool.map(lambda p: mec.describe_file(url[p]), vr_paths)))
vri = pd.DataFrame([{"path": p.split("/")[1], "subject": d["subject_id"], "identifier": d["identifier"],
                     "units": d["units"], "spikes": d["n_spikes"], "samples": d["n_samples"],
                     "contrast": d.get("contrast"), "gain": d.get("gain"), "ap_s": round(d.get("ap_seconds", np.nan)),
                     "ap_meta": d.get("ap_meta", "").split("found in ")[-1].replace("!", ""),
                     "exp_desc": d["experiment_description"]} for p, d in vrd.items()]).sort_values(["subject", "identifier", "path"])
print(vri.to_string(index=False))
print("\nSeries in head-fixed files:", sorted({tuple(d["behavior"]) for d in vrd.values()}))
print("Stated units:", sorted({tuple(sorted((k, str(v)) for k, v in d["units_of"].items())) for d in vrd.values()}))
print("Unit table columns:", sorted({tuple(d["unit_columns"]) for d in vrd.values()}))
print("Trial table columns:", sorted({tuple(d["trial_columns"]) for d in vrd.values() if d["has_trials"]}))
print("Electrode locations:", sorted({tuple(d["electrode_location"]) for d in vrd.values() if d["has_trials"]}))
print("experiment_description values:", sorted({d["experiment_description"] for d in vrd.values()}))

In [ ]:
# 1. Each small behavior file against the behavior+ecephys file with the same subject and identifier.
small = [p for p in vr_paths if "+ecephys" not in p]
pairs = []
for p in small:
    d = vrd[p]
    match = [q for q in vr_paths if "+ecephys" in q and vrd[q]["subject_id"] == d["subject_id"] and vrd[q]["identifier"] == d["identifier"]]
    pairs.append((p, match))
rows = []
for p, match in pairs:
    q = match[0]
    a, b = mec.spike_summary(url[p]), mec.spike_summary(url[q])
    with mec.open_file(url[p]) as f, mec.open_file(url[q]) as g:
        pos_same = np.array_equal(f["processing/behavior/Position/position/data"][:], g["processing/behavior/Position/position/data"][:])
        eye_same = np.array_equal(f["processing/behavior/EyeTracking/eye_position/data"][:], g["processing/behavior/EyeTracking/eye_position/data"][:])
    rows.append({"small file": p.split("/")[1], "ecephys files with same id": len(match), "units equal": a["units"] == b["units"],
                 "spike count equal": a["spikes"] == b["spikes"], "first/last spikes equal": np.array_equal(a["head"], b["head"]) and np.array_equal(a["tail"], b["tail"]),
                 "unit index equal": np.array_equal(a["index"], b["index"]), "position equal": pos_same, "eye equal": eye_same})
print(pd.DataFrame(rows).to_string(index=False))

# 2. Pairs of behavior+ecephys files from one subject and day.
eceph = [p for p in vr_paths if "+ecephys" in p]
by_day = {}
for p in eceph:
    by_day.setdefault((vrd[p]["subject_id"], vrd[p]["session_start"]), []).append(p)
for key, ps in by_day.items():
    if len(ps) < 2:
        continue
    n = int(vrd[ps[0]]["ap_seconds"] * 30000)
    same_raw = all(np.array_equal(mec.raw_samples(url[ps[0]], s0, 300), mec.raw_samples(url[ps[1]], s0, 300))
                   for s0 in (0, n // 2, n - 400))
    a, b = mec.spike_summary(url[ps[0]]), mec.spike_summary(url[ps[1]])
    print(f"\n{key}: {[p.split('/')[1] for p in ps]}")
    print(f"  identifiers {[vrd[p]['identifier'] for p in ps]}, raw AP lengths {[round(vrd[p]['ap_seconds']) for p in ps]} s, "
          f"raw samples equal at start, middle and end: {same_raw}")
    print(f"  units {a['units']} and {b['units']}; spikes {a['spikes']} and {b['spikes']}; "
          f"behaviour samples {[vrd[p]['n_samples'] for p in ps]}; first spike times {a['head'][:3].round(3)} and {b['head'][:3].round(3)}")

In [ ]:
paper_vr = [p for p in vr_paths if mec.is_vr_paper_session(vrd[p]["identifier"])]
# one file per recording: the small file where there is one, else the behavior+ecephys file
chosen = {}
for p in sorted(paper_vr, key=lambda p: "+ecephys" in p):
    chosen.setdefault((vrd[p]["subject_id"], vrd[p]["identifier"]), p)
vr_list = sorted(chosen.values())
vr = mec.load_many(mec.load_vr, [url[p] for p in vr_list], workers=11)
for p, s in zip(vr_list, vr):
    s["path"] = p
trials = {}
for s in vr:
    q = [e for e in eceph if vrd[e]["subject_id"] == s["subject"] and vrd[e]["identifier"] == s["identifier"]][0]
    trials[s["path"]] = mec.load_trials(url[q])
vinfo = pd.DataFrame([{"subject": s["subject"], "identifier": s["identifier"], "file": s["path"].split("/")[1],
                       "units": len(s["spikes"]), "minutes": round((s["t"][-1] - s["t"][0]) / 60, 1),
                       "dt_ms": round(1000 * np.median(np.diff(s["t"])), 3), "same_timestamps": s["same_timestamps"],
                       "trials": len(trials[s["path"]]["start_time"]),
                       "trials_gain_not_1": int(np.sum(trials[s["path"]]["running_wheel_gain"] != 1)),
                       "trials_contrast_not_100": int(np.sum(trials[s["path"]]["visual_contrast"] != 100))} for s in vr])
print(vinfo.to_string(index=False))

In [ ]:
print("Stated units:", sorted({tuple(sorted((k, str(v)) for k, v in s["units_of"].items())) for s in vr}))
rng_rows = []
for s in vr:
    r = {"session": f"{s['subject']} {s['identifier']}"}
    for k in ("B", "Bs", "Eh", "Ev", "Ehdot", "Evdot"):
        v = s[k][np.isfinite(s[k])]
        r[f"{k} p0.1"], r[f"{k} p99.9"] = np.percentile(v, 0.1).round(2), np.percentile(v, 99.9).round(2)
    r["eye NaN"] = round(float(np.mean(~np.isfinite(s["Eh"]))), 4)
    r["eye vel NaN"] = round(float(np.mean(~np.isfinite(s["Ehdot"]))), 4)
    rng_rows.append(r)
print(pd.DataFrame(rng_rows).to_string(index=False))
# speed against the change in track position, and eye velocity against the change in eye position
for s in vr[:3]:
    d = np.diff(s["B"]) / mec.DT
    ok = np.abs(d) < 300
    print(f"{s['subject']} {s['identifier']}: speed vs position change r = {np.corrcoef(d[ok], s['Bs'][1:][ok])[0, 1]:.3f}; "
          f"median ratio {np.median(s['Bs'][1:][ok] / np.where(np.abs(d[ok]) > 1, d[ok], np.nan)[np.abs(d[ok]) > 1]):.3f}")
for s in vr[:3]:
    k = np.exp(-0.5 * (np.arange(-35, 36) / 7) ** 2); k /= k.sum()     # Gaussian, sigma 7 samples = 140 ms
    for a, b in (("Eh", "Ehdot"), ("Ev", "Evdot")):
        e = pd.Series(s[a]).interpolate().bfill().ffill().to_numpy()
        de = np.convolve(np.gradient(e, mec.DT), k, mode="same")
        ok = np.isfinite(s[b])
        print(f"{s['subject']} {s['identifier']}: {b} against the smoothed derivative of {a}: r = {np.corrcoef(de[ok], s[b][ok])[0, 1]:.3f}, "
              f"slope {np.polyfit(de[ok], s[b][ok], 1)[0]:.3f}")

In [ ]:
# Clock checks. (1) Trial starts against the track position resets. (2) Population firing against
# running speed (head-fixed) or body speed (freely moving) at lags of -1 to 1 s. (3) Head-fixed: the
# eye position profile along the track, odd against even laps, true and with eye shifted by 60 s.
rows = []
for s in vr:
    tr = trials[s["path"]]
    wraps = s["t"][1:][np.diff(s["B"]) < -200]
    d = np.array([np.min(np.abs(wraps - t0)) for t0 in tr["start_time"][1:]])
    lag, c0, cbest = mec.population_lag(s["counts"], s["Bs"])
    lap = np.searchsorted(wraps, s["t"])
    bins = np.clip((s["B"] / 400 * 40).astype(int), 0, 39)
    def profile(e, parity):
        m = (lap % 2 == parity) & np.isfinite(e)
        return np.bincount(bins[m], e[m], 40) / np.maximum(np.bincount(bins[m], None, 40), 1)
    e = s["Eh"]
    true = np.corrcoef(profile(e, 0), profile(e, 1))[0, 1]
    sh = np.roll(e, int(60 / mec.DT))
    shifted = np.corrcoef(profile(sh, 0), profile(e, 1))[0, 1]
    rows.append({"session": f"{s['subject']} {s['identifier']}", "trial starts": len(tr["start_time"]), "resets": len(wraps),
                 "median |start - reset| ms": round(1000 * np.median(d), 1), "max ms": round(1000 * d.max(), 1),
                 "pop-speed lag s": lag, "r at 0": round(c0, 3), "r at best": round(cbest, 3),
                 "Eh profile odd-even r": round(true, 3), "with Eh shifted 60 s": round(shifted, 3)})
print(pd.DataFrame(rows).to_string(index=False))
lags = pd.DataFrame([dict(zip(("lag", "r0", "rbest"), mec.population_lag(s["counts"], s["Bs"]))) | {"subject": s["subject"]} for s in of])
print("\nFreely moving, population rate against body speed: lag of best correlation (s):")
print(lags.lag.describe().round(3).to_string())
print("sessions with |lag| > 0.2 s:", int((lags.lag.abs() > 0.2).sum()), "of", len(lags), "; median r at lag 0:", lags.r0.median().round(3))

**Data check, stated.** The pinned version and the draft hold the same files with the same sizes;
the draft differs in metadata only. The archive's summary counts 34 subjects; the files hold 25
freely moving mice and 9 head-fixed mice. Every file says `Mus musculus`.

*Freely moving.* 329 sessions of 25 mice with 1200 units, sampled at 50 Hz (20 ms). The stated
units (positions in metres with a conversion of 0.01, so the stored values are cm; speed in
cm/s; head direction in rad; azimuthal head velocity in rad/s) agree with the values: the speed
series follows the change of position and the head velocity series follows the change in head
direction (correlations printed above). The file's `experiment_description` gives one arena size;
the position extents show square arenas of about 100, 90, 70 and 50 cm and 75 by 37.5 cm
rectangles for the files that say 75 (the Methods list 100, 70, 50 and 70 by 35 cm arenas, and
not 91.4 or 75). The `session_start_time` of the Eggink et al. mice is in November 2019 for a study
published in 2014, so these dates are not recording dates. No file or unit repeats another. The
head-fixed checks are below the head-fixed cells.

*Head-fixed.* 30 files of 9 mice. The 10 small `behavior` files are copies of 10 of the
`behavior+ecephys` files (same units, spikes, position and eye traces), so each is one recording
deposited twice; they are counted once. Two pairs of `behavior+ecephys` files (npI4 2019-04-21 and
npJ5 2019-05-05) share the same raw recording, each with a different stretch of behaviour and
spike times; they are one recording deposited twice with different segments. Track position runs
0 to 400 cm (stated metres with conversion 0.01), speed is in cm/s, and the eye series are
labelled metres and unknown, though their ranges are those of Figure 4f to i in percent of the eye
width per sample and per second; the files are read in those units. The stated
`experiment_description` "trial contrast: 100" is in every file, including those whose trial
table has contrasts 0 to 50. The electrode locations are "unknown", so the hemisphere of each
recording is not in the files. Trials start at the resets of the track position, the population
rate correlates best with running speed at a lag near zero in every session, and the eye position
profile along the track repeats between odd and even laps and not when shifted, so the streams
are consistent with one clock. A lag of a few samples between eye and spikes could not be
excluded by these checks.

## vr-dataset-size

> The head-fixed dataset comprises 2861 MEC units recorded with Neuropixels probes in 11 sessions from 8 mice.

In [ ]:
paper_set = vinfo
print("Sessions whose identifier contains 'baseline', one file per recording:")
print(paper_set[["subject", "identifier", "file", "units"]].to_string(index=False))
print(f"\nunits {paper_set.units.sum()}, sessions {len(paper_set)}, mice {paper_set.subject.nunique()}  (paper 2861, 11, 8)")
others = vri[~vri.identifier.str.contains("baseline")]
print("\nThe other head-fixed recordings (identifiers without 'baseline'):")
print(others[["subject", "identifier", "path", "units"]].to_string(index=False))
dedup = others.drop_duplicates(["subject", "ap_meta"])
print(f"units in them, each raw recording once: {dedup.units.sum()} in {len(dedup)} recordings; "
      f"all head-fixed recordings: {paper_set.units.sum() + dedup.units.sum()}")
print("\nTrial table of the 11 sessions: trials with gain not 1 or contrast not 100, per session:")
print(paper_set[["subject", "identifier", "trials", "trials_gain_not_1", "trials_contrast_not_100"]].to_string(index=False))

**Result: PLACEHOLDER**

## of-cohorts-by-count

> The freely moving dataset comprises 1200 MEC cells; Cohort 1 (6 mice) contributed 179 and Cohort 2 (20 mice) 1021, of which 228 unpublished (5 mice), 122 from Campbell et al. (2), 389 from Eggink et al. (7) and 282 from Munn et al. (6).

In [ ]:
g = ofi.groupby("group").agg(mice=("subject", "nunique"), units=("units", "sum"), sessions=("units", "size"),
                             arenas=("arena", lambda a: sorted(set(a))))
g["Methods mice"] = [mec.METHODS_GROUPS[i][0] for i in g.index]
g["Methods cells"] = [mec.METHODS_GROUPS[i][1] for i in g.index]
print(g.to_string())
print(f"\nTotal: {ofi.subject.nunique()} mice, {ofi.units.sum()} units; Methods 26 mice, 1200 cells; "
      f"Cohort 1 {g.loc['Cohort 1', 'units']} units in {g.loc['Cohort 1', 'mice']} mice")
print("\nFeatures per subject (identifier pattern, arenas, dates):")
feat = ofi.groupby("subject").agg(group=("group", "first"), example_ids=("identifier", lambda s: list(s)[:2]),
                                  arenas=("arena", lambda a: sorted(set(a))), first=("date", "min"), last=("date", "max"))
print(feat.sort_values("group").to_string())
# other subsets of subjects that sum to the counts of the two smallest groups
units = ofi.groupby("subject").units.sum()
import itertools
pairs = [c for c in itertools.combinations(units.index, 2) if units[list(c)].sum() == 122]
print("\nPairs of subjects with 122 units:", pairs)

**Result: PLACEHOLDER**

## fig1b-ahv-percentiles

> Across 25 mice the distribution of azimuthal head velocity has 2.5th, 25th, 50th, 75th and 97.5th percentiles of -389.9, -37.7, -0.1, 36.4 and 385.1 deg/s.

In [ ]:
def pct_table(sessions, keep):
    rows = []
    for s in sessions:
        v = s["Hadot"]
        m = np.isfinite(v)
        if keep:
            vals = {"x": s["x"], "y": s["y"], "Bs": s["Bs"], "Hadot": s["Hadot"]}
            m = mec.keep_mask(vals, ["x", "y", "Bs", "Hadot"], mec.OF_KEEP)
        rows.append(dict(zip(["p2.5", "p25", "p50", "p75", "p97.5"], np.percentile(v[m], [2.5, 25, 50, 75, 97.5]))) | {"subject": s["subject"]})
    per_mouse = pd.DataFrame(rows).groupby("subject").mean()
    return per_mouse, per_mouse.mean(), per_mouse.sem()
paper = pd.Series({"p2.5": -389.9, "p25": -37.7, "p50": -0.1, "p75": 36.4, "p97.5": 385.1})
tol = pd.Series({"p2.5": 34.7, "p25": 3.9, "p50": 0.3, "p75": 3.8, "p97.5": 34.7})
for name, keep in (("(a) all samples", False), ("(b) samples kept for the model", True)):
    per_mouse, mean, sem = pct_table(of, keep)
    out = pd.DataFrame({"here mean": mean, "here sem": sem, "paper": paper, "tolerance": tol,
                        "within": (mean - paper).abs() <= tol})
    print(f"Reading {name}, {len(per_mouse)} mice:")
    print(out.round(2).to_string(), "\n")
    if not keep:
        print("Per mouse (deg/s):")
        print(per_mouse.round(1).to_string(), "\n")

**Result: PLACEHOLDER**

## fig4fghi-eye-percentiles

> Head-fixed mice moved their eyes considerably (percentiles of eye position and velocity across 8 mice, Figure 4f to i).

In [ ]:
names = {"Eh": "horizontal position", "Ev": "vertical position", "Ehdot": "horizontal velocity", "Evdot": "vertical velocity"}
paper_eye = {"Eh": [-3.50, -0.97, -0.11, 0.83, 5.28], "Ev": [-2.74, -0.35, -0.035, 0.28, 1.60],
             "Ehdot": [-6.19, -0.58, -0.11, 0.36, 6.87], "Evdot": [-2.83, -0.29, 0.013, 0.31, 3.15]}
sem_eye = {"Eh": [0.61, 0.27, 0.032, 0.26, 0.87], "Ev": [0.64, 0.11, 0.11, 0.11, 0.21],
           "Ehdot": [0.74, 0.091, 0.029, 0.12, 0.66], "Evdot": [0.55, 0.025, 0.010, 0.031, 0.68]}
q = [1, 25, 50, 75, 99]
rows, within = [], 0
per_session = []
for s in vr:
    for k in names:
        v = s[k][np.isfinite(s[k])]
        per_session.append({"subject": s["subject"], "session": s["identifier"], "var": k, **dict(zip([f"p{x}" for x in q], np.percentile(v, q)))})
ps = pd.DataFrame(per_session)
for k in names:
    pm = ps[ps["var"] == k].groupby("subject")[[f"p{x}" for x in q]].mean()
    for i, x in enumerate(q):
        m, e = pm[f"p{x}"].mean(), pm[f"p{x}"].sem()
        ok = abs(m - paper_eye[k][i]) <= sem_eye[k][i]
        within += ok
        rows.append({"variable": k, "percentile": x, "here mean": m, "here sem": e, "paper": paper_eye[k][i],
                     "tolerance": sem_eye[k][i], "within": ok})
print(f"{ps.subject.nunique()} mice, {ps.session.nunique()} sessions")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"\nwithin tolerance: {within} of 20")
print("\nPer session (all finite samples):")
print(ps.round(2).to_string(index=False))

**Result: PLACEHOLDER**

## figs12-eye-behavior-mi

> Eye position and eye velocity on both axes were more correlated than chance with body position and running speed on the virtual track.

In [ ]:
rng = np.random.default_rng(53)
rows = []
for s in vr:
    vals = {k: s[k] for k in mec.VR_VARS}
    keep = mec.keep_mask(vals, mec.VR_VARS, mec.VR_KEEP)
    for e in ("Eh", "Ev", "Ehdot", "Evdot"):
        for b in ("B", "Bs"):
            x, y = s[e][keep], s[b][keep]
            mi = mec.mutual_information(x, y)
            null = np.array([mec.mutual_information(rng.permutation(x), y) for _ in range(100)])
            rows.append({"session": f"{s['subject']} {s['identifier']}", "pair": f"{e}-{b}", "MI bits": mi,
                         "null p95": np.percentile(null, 95), "null max": null.max(), "exceeds": mi > np.percentile(null, 95)})
mi = pd.DataFrame(rows)
summary = mi.groupby("pair").agg(sessions_exceeding=("exceeds", "sum"), MI_median=("MI bits", "median"),
                                 MI_min=("MI bits", "min"), null_p95_median=("null p95", "median"))
print(summary.round(4).to_string())
print(f"\npairs exceeding in at least 6 of {len(vr)} sessions: {(summary.sessions_exceeding >= 6).sum()} of 8")
print("\nPer session:")
print(mi.pivot(index="session", columns="pair", values="MI bits").round(3).to_string())

**Result: PLACEHOLDER**

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

PLACEHOLDER